<div dir="rtl" lang="he" style="text-align: right;">

# מפגש 23 — GridSearchCV וקריאת תוצאות
2 שעות הוראה ותרגול ו־1 שעות פרויקט; יחד 135 דקות.

## סיפור המסגרת
דודו משווה שלושה ערכי K. נרשום מראש את המועמדים ונבקש מחיפוש שיטתי לבדוק אותם באותה דרך.

## מטרות
- להגדיר param_grid.
- לקרוא cv_results_.
- להסביר best_params_ ו־refit.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## הנחיות למורה
זהו פתרון מתועד למעבדה. מבדילים בין תוצאה נכונה לבין נימוק נכון, ובודקים שהקוד תואם לנתוני התלמיד. המקורות הישנים נשמרו ללא שינוי.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## פתרון חלק 1
הפתרון הבא תואם לנתוני התרגול העצמאי, ולא לנתוני הדוגמה הפתורה.

</div>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df=pd.read_csv(url)
data=df.dropna(subset=["species"]).copy()
numeric=["flipper_length_mm","body_mass_g"]
categorical=["island"]
X=data[numeric+categorical].copy()
y=data["species"].copy()
X_train,X_test,y_train,y_test=train_test_split(
 X,y,test_size=.25,random_state=17,stratify=y)
print("Train:",X_train.shape,"Test:",X_test.shape)

<div dir="rtl" lang="he" style="text-align: right;">

## הכנה משולבת
המספריים עוברים השלמת חציון ותקנון; island קידוד שמי ושכיח. אין להפעיל את prepare.fit על כל X לפני חלוקה או על כל Train לפני CV.

</div>

In [ ]:
numeric_pipeline=Pipeline([
 ("fill",SimpleImputer(strategy="median")),
 ("scale",StandardScaler())
])
categorical_pipeline=Pipeline([
 ("fill",SimpleImputer(strategy="most_frequent")),
 ("encode",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])
prepare=ColumnTransformer([
 ("numeric",numeric_pipeline,numeric),
 ("categorical",categorical_pipeline,categorical)
],remainder="drop")
pipe=Pipeline([
 ("prepare",prepare),
 ("knn",KNeighborsClassifier(weights="uniform",metric="euclidean"))
])

<div dir="rtl" lang="he" style="text-align: right;">

## חיפוש ותוצאות
שלושה מועמדים×חמישה קיפולים=15 התאמות CV ועוד אחת ל־ refit. cv_results_ מתעד גם מועמדים שלא נבחרו; best_params_ אינו סכום כל הניסויים.

</div>

In [ ]:
# ההגדרות נבחרו מראש,בלי שימוש בציוני Test
param_grid={"knn__n_neighbors":[3,5,9]}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
clf=GridSearchCV(
 estimator=pipe,param_grid=param_grid,cv=cv,scoring="f1_macro",
 refit=True,return_train_score=True,n_jobs=1,error_score="raise"
)
clf.fit(X_train,y_train)
print("Best params:",clf.best_params_)
print("Best CV macro-F1:",clf.best_score_)
cv_results=pd.DataFrame(clf.cv_results_)
display(cv_results[[
 "param_knn__n_neighbors","mean_train_score","mean_test_score",
 "std_test_score","rank_test_score"
]].sort_values("rank_test_score"))

<div dir="rtl" lang="he" style="text-align: right;">

## הערכה
רק לאחר הבחירה משתמשים ב־ Test לדיווח. משווים את קו הבסיס והמודל באותו macro-F1. כמה מדדי דיווח על אותן תחזיות אינם רשות לחזור לבחור מודל לפי Test.

</div>

In [ ]:
# בודקים קו בסיס באותו מדד CV
baseline=DummyClassifier(strategy="most_frequent")
baseline_cv=cross_val_score(baseline,X_train,y_train,cv=cv,scoring="f1_macro")
baseline.fit(X_train,y_train)
print("Baseline CV macro-F1:",baseline_cv.mean())

# refit=True כבר התאים את המודל הנבחר לכל Train
predictions=clf.predict(X_test)
baseline_predictions=baseline.predict(X_test)
print("Final model Test macro-F1:",f1_score(y_test,predictions,average="macro"))
print("Baseline Test macro-F1:",f1_score(y_test,baseline_predictions,average="macro"))
print("Final model Test Accuracy:",accuracy_score(y_test,predictions))
print(classification_report(y_test,predictions,zero_division=0))

<div dir="rtl" lang="he" style="text-align: right;">

## דגשים להגנה
param_grid מגדיר מועמדים ולא פרמטרים שנלמדו. עבור 3 ערכי K וחמישה קיפולים יש 15 התאמות CV; refit=True מוסיף התאמה 16 על כל Train. mean_test_score הוא ממוצע ציוני האימות הפנימיים ולא ציון Test החיצוני. best_params_ מציג את המועמד שנבחר לפי המדד; best_score_ הוא ממוצע CV שלו. std_test_score הוא פיזור תיאורי של קיפולים, לא רווח סמך. בשוויון GridSearchCV בוחר את המועמד הראשון בדירוג סדר החיפוש; אין הבטחה לעליונות מובהקת.
לא מעבירים X_test ל־ clf.fit; לא בוחרים לפי mean_train_score; לא מפרשים test_score הפנימי כמבחן חיצוני; לא מוחקים __ בשם הפרמטר; לא מתאימים Encoder לפני CV. מודל על אי ומינים אינו מבטיח הכללה לאי חדש. אין הבטחת דיוק נדרש בתרגיל.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## תשובה לכרטיס היציאה
הראשון הוא בחירת היפר־פרמטרים בחיפוש. הממוצע הוא פרמטר הכנה שנלמד מנתוני אימון.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## משוב דידקטי
בדקו בנפרד: בחירת פעולה, תקינות קוד, פירוש פלט, יחידות ומגבלה. שאלו מה ישתנה בקלט חדש. אם יש קושי, חזרו לדוגמה קטנה לפני הגדלת המאגר. זהו משוב מקומי, לא מחוון ניקוד רשמי.

</div>